In [0]:
# Databricks notebook source

# ==========================================================
# Utility Notebook
#
# Notebook : 01_watermark_utils
#
# Purpose:
# Read and update processing watermarks
#
# ==========================================================

from pyspark.sql import functions as F
from pyspark.sql import Row
from delta.tables import DeltaTable
from datetime import datetime, UTC

CATALOG = "crypto_pipeline"

WATERMARK_TABLE = f"{CATALOG}.meta.watermarks"

spark.sql(f"USE CATALOG {CATALOG}")

# ==========================================================
# Create Watermark Table
# ==========================================================

spark.sql(f"""

CREATE TABLE IF NOT EXISTS {WATERMARK_TABLE}

(

table_name STRING,

last_processed_timestamp TIMESTAMP,

last_updated TIMESTAMP

)

USING DELTA

""")

# ==========================================================
# Function: Get Watermark
# ==========================================================

def get_watermark(table_name):

    df = (

        spark.table(WATERMARK_TABLE)

        .filter(

            F.col("table_name") == table_name

        )

    )

    if df.count() == 0:

        return None

    return df.collect()[0]["last_processed_timestamp"]

# ==========================================================
# Function: Update Watermark
# ==========================================================

def update_watermark(table_name, timestamp):

    delta = DeltaTable.forName(

        spark,

        WATERMARK_TABLE

    )

    source = spark.createDataFrame(

        [

            Row(

                table_name=table_name,

                last_processed_timestamp=timestamp,

                last_updated=datetime.now(UTC)

            )

        ]

    )

    (

        delta.alias("t")

        .merge(

            source.alias("s"),

            "t.table_name=s.table_name"

        )

        .whenMatchedUpdate(

            set={

                "last_processed_timestamp":"s.last_processed_timestamp",

                "last_updated":"s.last_updated"

            }

        )

        .whenNotMatchedInsert(

            values={

                "table_name":"s.table_name",

                "last_processed_timestamp":"s.last_processed_timestamp",

                "last_updated":"s.last_updated"

            }

        )

        .execute()

    )

# ==========================================================
# Function: Show Watermarks
# ==========================================================

def show_watermarks():

    display(

        spark.table(

            WATERMARK_TABLE

        )

    )

print("Watermark utilities loaded.")

Watermark utilities loaded.
